In [26]:
import os

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
APP_DIR = os.path.join(PROJECT_DIR, "app")
os.makedirs(APP_DIR, exist_ok=True)


app_code = r'''
import os
import joblib
import numpy as np
import pandas as pd
import streamlit as st

# Optional SHAP (if installed)
try:
    import shap
    import matplotlib.pyplot as plt
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False

# ---------------- PAGE CONFIG ----------------
st.set_page_config(page_title="Smart Diabetes Risk Predictor", layout="wide")

st.title("🩺 Smart Diabetes Risk Predictor")
st.write("An interpretable ML-based decision support tool for diabetes risk prediction using lifestyle and clinical data.")

# ---------------- PATHS ----------------
BASE_DIR = os.path.dirname(__file__)
PROJECT_DIR = os.path.dirname(BASE_DIR)

PIPELINE_PATH = os.path.join(PROJECT_DIR, "models", "diabetes_risk_pipeline.pkl")
THRESH_PATH = os.path.join(PROJECT_DIR, "models", "best_threshold.pkl")

if not os.path.exists(PIPELINE_PATH) or not os.path.exists(THRESH_PATH):
    st.error("❌ Pipeline or threshold file not found in /models/")
    st.write("Expected:", PIPELINE_PATH)
    st.write("Expected:", THRESH_PATH)
    st.stop()

pipeline = joblib.load(PIPELINE_PATH)
best_threshold = float(joblib.load(THRESH_PATH))

# ---------------- INPUT OPTIONS (SAFE DEFAULTS) ----------------
ETHNICITY_OPTIONS = ["White", "Asian", "Black", "Hispanic", "Other"]
PHYSICAL_OPTIONS = ["High" , "Low" , "Moderate"]
ALCOHOL_OPTIONS = ["None", "Moderate", "Heavy"]
SMOKING_OPTIONS = ["Never", "Former", "Current"]

# ---------------- SIDEBAR INPUTS ----------------
st.sidebar.header("User Inputs")

age = st.sidebar.number_input(
    "Age (years)",
    min_value=0,
    max_value=120,
    value=30,
    step=1
)

bmi = st.sidebar.number_input(
    "BMI",
    min_value=0.0,
    max_value=45.0,
    value=20.0
)

waist = st.sidebar.number_input(
    "Waist Circumference (cm)",
    min_value=0.0,
    max_value=60.0,
    value=30.0
)

sex = st.sidebar.selectbox("Sex", ["Female", "Male"])

diet_cal = st.sidebar.number_input(
    "Dietary Intake Calories (kcal/day)",
    min_value=0.0,
    value=2000.0
)

fh = st.sidebar.selectbox(
    "Family History of Diabetes",
    ["No", "Yes"]
)

st.sidebar.subheader("Lifestyle Inputs")

ethnicity = st.sidebar.selectbox(
    "Ethnicity",
    ETHNICITY_OPTIONS
)

physical = {
    "High": "Low",
    "Low": "High",
    "Moderate": "Moderate"
}[st.sidebar.selectbox(
    "Physical Activity Level",
    ["High", "Low", "Moderate"]
)]

alcohol = st.sidebar.selectbox(
    "Alcohol Consumption",
    ALCOHOL_OPTIONS
)

smoking = st.sidebar.selectbox(
    "Smoking Status",
    SMOKING_OPTIONS
)

# ---------------- BUILD RAW INPUT DATAFRAME ----------------

X_input = pd.DataFrame([{
    "Age": age,
    "BMI": bmi,
    "Waist_Circumference": waist,
    "Sex": sex,
    "Dietary_Intake_Calories": diet_cal,
    "Family_History_of_Diabetes": 1 if fh == "Yes" else 0,
    "Ethnicity": ethnicity,
    "Physical_Activity_Level": physical,
    "Alcohol_Consumption": alcohol,
    "Smoking_Status": smoking
}])

# ---------------- PREDICTION ----------------
st.subheader("Prediction Result")

try:
    proba = float(pipeline.predict_proba(X_input)[0, 1])

except Exception as e:
    st.error("❌ Prediction failed due to column mismatch or preprocessing mismatch.")
    st.write("Error:", str(e))
    st.write("Input columns:", list(X_input.columns))
    st.stop()

pred = int(proba > best_threshold)

# Risk category
if proba > best_threshold:
    risk = "High Risk"
else:
    risk = "Low Risk"

c1, c2, c3 = st.columns(3)

c1.metric("Predicted Class (0/1)", pred)
c2.metric("Probability (Diabetes=1)", f"{proba:.2f}")
c3.metric("Risk Category", risk)

st.caption(f"Decision threshold used: {best_threshold:.2f}")

# ---------------- RECOMMENDATIONS ----------------
st.subheader("Basic Recommendations")

recs = []

if bmi >= 25:
    recs.append("BMI is elevated; consider healthy diet and regular physical activity.")

if waist >= 94 and sex == "Male":
    recs.append("Waist circumference is high for males; central obesity is a strong diabetes risk factor.")

if waist >= 80 and sex == "Female":
    recs.append("Waist circumference is high for females; central obesity increases diabetes risk.")

if fh == "Yes":
    recs.append("Family history increases risk; periodic screening is recommended.")

if physical == "High":
    recs.append("Physical activity is low; increasing daily movement may reduce risk.")

if smoking == "Current":
    recs.append("Current smoking is associated with metabolic risk; cessation is recommended.")

if alcohol.lower() == "heavy":
    recs.append("Heavy alcohol consumption may affect metabolic health; reduce intake based on guidance.")

if not recs:
    recs.append("Maintain healthy lifestyle habits and schedule regular health check-ups.")

for r in recs:
    st.write("• " + r)
'''

app_path = os.path.join(APP_DIR, "app.py")
with open(app_path, "w", encoding="utf-8") as f:
    f.write(app_code)

print("✅ Streamlit app written to:", app_path)


✅ Streamlit app written to: C:\Users\LENOVO\diabetes_project\app\app.py
